In [1]:
import os
import numpy as np

import mlflow
from mlflow.tracking import MlflowClient

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [ ]:
print(mlflow.__version__)

In [6]:
print("MLFLOW_TRACKING_URI =", os.environ.get("MLFLOW_TRACKING_URI", "NOT SET"))
print("mlflow.get_tracking_uri() =", mlflow.get_tracking_uri())

MLFLOW_TRACKING_URI = http://158.160.184.127:5000
mlflow.get_tracking_uri() = http://158.160.184.127:5000


In [5]:
mlflow.set_tracking_uri("http://158.160.184.127:5000")
mlflow.set_experiment("final-task")

2026/03/16 20:06:41 INFO mlflow.tracking.fluent: Experiment with name 'final-task' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/mlflow/artifacts/1', creation_time=1773691601954, experiment_id='1', last_update_time=1773691601954, lifecycle_stage='active', name='final-task', tags={}, workspace='default'>

In [7]:
data = load_diabetes(scaled=False)
X, y = data.data, data.target
feature_names = data.feature_names
feature_names


['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipe = make_pipeline(
    StandardScaler(),
    RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
)

with mlflow.start_run() as run:
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("random_state", 42)

    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    rmse = mean_squared_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    model_info = mlflow.sklearn.log_model(
        sk_model=pipe,
        artifact_path="model",
        registered_model_name="diabets",
        input_example=X_test[:2],
    )

    run_id = run.info.run_id

run_id, rmse, r2, model_info.model_uri


2026/03/16 20:15:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/16 20:15:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'diabets' already exists. Creating a new version of this model...
2026/03/16 20:16:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: diabets, version 2
Created version '2' of model 'diabets'.


🏃 View run selective-ant-867 at: http://158.160.184.127:5000/#/experiments/1/runs/fefdd3bfff674a369db4879167356843
🧪 View experiment at: http://158.160.184.127:5000/#/experiments/1


('fefdd3bfff674a369db4879167356843',
 3016.354074531835,
 0.4306780519496508,
 'models:/m-d08ba3c820704eec86965a38f8baf746')

In [13]:
client = MlflowClient()
versions = client.search_model_versions("name='diabets'")

latest = sorted(versions, key=lambda v: int(v.version))[-1]
latest.version, latest.current_stage, latest.status, latest.run_id


('2', 'None', 'READY', 'fefdd3bfff674a369db4879167356843')

In [14]:
version = latest.version

model = mlflow.sklearn.load_model(f"models:/diabets/{version}")



In [21]:
import shutil
from pathlib import Path
export_path = "./model"

mlflow.sklearn.save_model(model, path=str(export_path))

2026/03/16 20:21:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
